# LeetCode 1688 — Count of Matches in Tournament bằng Python

**Link đề:** https://leetcode.com/problems/count-of-matches-in-tournament/
**Độ khó:** Easy · **Chủ đề:** Math, Simulation

```python
class Solution:
    def numberOfMatches(self, n: int) -> int:
        ...
```

## Mục tiêu học
1. Đọc kỹ quy tắc giải đấu và **mô phỏng** đúng từng vòng (chẵn / lẻ).
2. Nắm công cụ Python: `%` (chia lấy dư), `//` (chia nguyên), vòng `while`, đệ quy.
3. Từ mô phỏng `O(log n)` tìm ra **insight bất biến**: mỗi trận loại đúng 1 đội ⇒ đáp án là `n - 1` (`O(1)`).
4. Trace từng vòng, chứng minh tính đúng bằng lập luận "đếm theo đội bị loại".
5. Biết khi nào nên mô phỏng, khi nào nên tìm công thức; tự giải biến thể.

## 1. Đề bài và ví dụ

Có `n` đội tham gia giải đấu với luật:
- Nếu số đội hiện tại **chẵn**: các đội ghép cặp, đá `n / 2` trận, `n / 2` đội đi tiếp.
- Nếu số đội hiện tại **lẻ**: một đội được **đặc cách** đi tiếp, số còn lại ghép cặp: đá `(n - 1) / 2` trận, `(n - 1) / 2 + 1` đội đi tiếp.

Trả về **tổng số trận** đã đá cho đến khi chọn được nhà vô địch.

| `n` | Diễn biến | Tổng trận |
|---:|---|---:|
| `7` | 7 → (3 trận) → 4 → (2) → 2 → (1) → 1 | `6` |
| `14` | 14 → (7) → 7 → (3) → 4 → (2) → 2 → (1) → 1 | `13` |

**Constraints:** `1 <= n <= 200`.

Chú ý ca `n = 1`: chỉ một đội, đã là vô địch, **0 trận**.

### 1.1. Tính tay một vòng

Với `n = 7` (lẻ): 1 đội đặc cách, 6 đội còn lại ghép thành 3 cặp → 3 trận; đi tiếp = 3 đội thắng + 1 đội đặc cách = 4.

In [1]:
n = 7
matches = (n - 1) // 2
advance = (n - 1) // 2 + 1
print("n =", n, "| trận vòng này =", matches, "| đi tiếp =", advance)

n = 4
print("n =", n, "| trận vòng này =", n // 2, "| đi tiếp =", n // 2)

n = 7 | trận vòng này = 3 | đi tiếp = 4
n = 4 | trận vòng này = 2 | đi tiếp = 2


## 2. Nền tảng Python cần biết

### 2.1. `%` — kiểm tra chẵn / lẻ

`n % 2` là số dư khi chia `n` cho 2: bằng `0` nếu chẵn, `1` nếu lẻ.

In [2]:
for n in [7, 14, 1, 2]:
    print(n, "% 2 =", n % 2, "->", "chẵn" if n % 2 == 0 else "lẻ")

7 % 2 = 1 -> lẻ
14 % 2 = 0 -> chẵn
1 % 2 = 1 -> lẻ
2 % 2 = 0 -> chẵn


### 2.2. `//` — chia lấy phần nguyên (khác `/`)

`/` luôn cho số thực (`float`), `//` cho số nguyên (làm tròn xuống). Số trận phải là số nguyên nên ta dùng `//`. Mẹo: với `n` lẻ, `n // 2 == (n - 1) // 2`.

In [3]:
print(7 / 2, 7 // 2, (7 - 1) // 2)
print(14 / 2, 14 // 2)
print(type(14 / 2), type(14 // 2))

3.5 3 3
7.0 7
<class 'float'> <class 'int'>


### 2.3. Biểu thức điều kiện một dòng và `(n + 1) // 2`

`a if cond else b` chọn giá trị theo điều kiện. Số đội đi tiếp trong **cả hai** trường hợp đều bằng `(n + 1) // 2` (làm tròn lên của `n / 2`): chẵn → `n/2`, lẻ → `(n-1)/2 + 1 = (n+1)/2`.

In [4]:
for n in [7, 8, 1, 2, 13]:
    advance_rule = n // 2 if n % 2 == 0 else (n - 1) // 2 + 1
    print(n, "-> theo luật:", advance_rule, "| công thức (n+1)//2:", (n + 1) // 2)

7 -> theo luật: 4 | công thức (n+1)//2: 4
8 -> theo luật: 4 | công thức (n+1)//2: 4
1 -> theo luật: 1 | công thức (n+1)//2: 1
2 -> theo luật: 1 | công thức (n+1)//2: 1
13 -> theo luật: 7 | công thức (n+1)//2: 7


### 2.4. Vòng `while` — lặp khi điều kiện còn đúng

`while n > 1:` lặp cho đến khi chỉ còn 1 đội. Mỗi vòng **phải** làm `n` giảm, nếu không vòng lặp chạy mãi.

In [5]:
n = 10
round_no = 0
while n > 1:
    round_no += 1
    n = (n + 1) // 2
    print("sau vòng", round_no, "còn", n, "đội")

sau vòng 1 còn 5 đội
sau vòng 2 còn 3 đội
sau vòng 3 còn 2 đội
sau vòng 4 còn 1 đội


### 2.5. Đệ quy — hàm tự gọi chính nó

Đệ quy gồm **trường hợp cơ sở** (dừng) và **bước đệ quy** (bài toán nhỏ hơn). Ví dụ tính tổng 1..k:

In [6]:
def sum_to(k):
    if k == 0:              # trường hợp cơ sở
        return 0
    return k + sum_to(k - 1)  # bước đệ quy: bài toán nhỏ hơn

print(sum_to(4))   # 4 + 3 + 2 + 1 + 0 = 10

10


## 3. Cách trực giác — mô phỏng đúng như đề

Ta giữ `teams` (số đội còn lại) và `total` (tổng trận). Mỗi vòng:
1. Nếu `teams` chẵn: `matches = teams // 2`, `teams = teams // 2`.
2. Nếu lẻ: `matches = (teams - 1) // 2`, `teams = (teams - 1) // 2 + 1`.
3. `total += matches`.

Dừng khi `teams == 1`.

In [7]:
def matches_simulation(n: int) -> int:
    teams = n
    total = 0
    while teams > 1:
        if teams % 2 == 0:
            matches = teams // 2
            teams = teams // 2
        else:
            matches = (teams - 1) // 2
            teams = (teams - 1) // 2 + 1
        total += matches
    return total

print(matches_simulation(7))    # 6
print(matches_simulation(14))   # 13
print(matches_simulation(1))    # 0

6
13
0


### Gọn hơn: một công thức cho cả hai trường hợp

Số trận mỗi vòng = `teams // 2` (cả chẵn lẫn lẻ), số đội đi tiếp = `(teams + 1) // 2`.

In [8]:
def matches_simulation_compact(n: int) -> int:
    total = 0
    while n > 1:
        total += n // 2
        n = (n + 1) // 2
    return total

print([matches_simulation_compact(n) for n in [1, 2, 3, 7, 14, 200]])

[0, 1, 2, 6, 13, 199]


### Độ phức tạp của mô phỏng

Mỗi vòng số đội giảm gần một nửa → số vòng ≈ `⌈log₂ n⌉`. Time `O(log n)`, Space `O(1)`. Với `n ≤ 200` chỉ tối đa 8 vòng — quá đủ nhanh. Nhưng hãy nhìn lại các kết quả: `7 → 6`, `14 → 13`, `200 → 199`… có gì đáng ngờ?

In [9]:
for n in range(1, 16):
    print(f"n = {n:>2} -> {matches_simulation(n):>2} trận | n - 1 = {n - 1}")

n =  1 ->  0 trận | n - 1 = 0
n =  2 ->  1 trận | n - 1 = 1
n =  3 ->  2 trận | n - 1 = 2
n =  4 ->  3 trận | n - 1 = 3
n =  5 ->  4 trận | n - 1 = 4
n =  6 ->  5 trận | n - 1 = 5
n =  7 ->  6 trận | n - 1 = 6
n =  8 ->  7 trận | n - 1 = 7
n =  9 ->  8 trận | n - 1 = 8
n = 10 ->  9 trận | n - 1 = 9
n = 11 -> 10 trận | n - 1 = 10
n = 12 -> 11 trận | n - 1 = 11
n = 13 -> 12 trận | n - 1 = 12
n = 14 -> 13 trận | n - 1 = 13
n = 15 -> 14 trận | n - 1 = 14


## 4. Nút thắt và insight

Mô phỏng đã đủ nhanh, nhưng **dữ liệu gợi ý một quy luật**: đáp án luôn bằng `n - 1`. Tại sao?

**Đổi góc nhìn:** thay vì đếm trận theo vòng, hãy đếm theo **đội bị loại**.
- Mỗi trận có đúng 1 đội thua → bị loại. Đội đặc cách không đá nên không bị loại.
- Không có cách nào bị loại mà không thua một trận.
⇒ **Số trận = số đội bị loại.**
- Kết thúc giải: còn đúng 1 đội (nhà vô địch) ⇒ đã loại `n - 1` đội.

⇒ **Số trận = `n - 1`**, bất kể luật ghép cặp hay đặc cách thế nào!

In [10]:
# Kiểm chứng: đếm số đội bị loại trong mô phỏng
def eliminated_count(n):
    teams, eliminated = n, 0
    while teams > 1:
        losers = teams // 2       # mỗi trận 1 đội thua
        eliminated += losers
        teams -= losers
    return eliminated

for n in [1, 7, 14, 200]:
    print(n, "-> số đội bị loại:", eliminated_count(n), "| số trận:", matches_simulation(n))

1 -> số đội bị loại: 0 | số trận: 0
7 -> số đội bị loại: 6 | số trận: 6
14 -> số đội bị loại: 13 | số trận: 13
200 -> số đội bị loại: 199 | số trận: 199


### 4.1. Thí nghiệm: đổi luật ghép cặp, đáp án vẫn là `n - 1`

Để tin insight "không phụ thuộc cách ghép cặp", ta mô phỏng một giải knock-out **ngẫu nhiên**: mỗi bước chọn bất kỳ 2 đội còn lại đá với nhau, đội thua bị loại (không cần chia vòng). Dùng `random.sample` để chọn 2 đội và `list.remove` để xoá đội thua.

In [11]:
import random

def random_knockout(n, seed):
    rng = random.Random(seed)          # bộ sinh ngẫu nhiên riêng, có seed để lặp lại được
    alive = list(range(n))             # danh sách đội còn sống
    matches = 0
    while len(alive) > 1:
        a, b_ = rng.sample(alive, 2)   # chọn 2 đội khác nhau
        loser = rng.choice([a, b_])
        alive.remove(loser)
        matches += 1
    return matches

for n in [1, 7, 14, 50]:
    results = {random_knockout(n, seed) for seed in range(20)}
    print(f"n = {n:>2}: số trận qua 20 lần ghép ngẫu nhiên = {results}")

n =  1: số trận qua 20 lần ghép ngẫu nhiên = {0}
n =  7: số trận qua 20 lần ghép ngẫu nhiên = {6}
n = 14: số trận qua 20 lần ghép ngẫu nhiên = {13}
n = 50: số trận qua 20 lần ghép ngẫu nhiên = {49}


Dù ghép thế nào, tập kết quả chỉ có **một giá trị** `n - 1`. Đây là dấu hiệu của một **bất biến**: mỗi trận làm số đội còn sống giảm đúng 1.

## 5. Lời giải chính — công thức `n - 1`

Ghép lại: không cần vòng lặp, không cần phân biệt chẵn lẻ.

In [12]:
def matches_formula(n: int) -> int:
    return n - 1

print(matches_formula(7), matches_formula(14), matches_formula(1))

6 13 0


## 6. TRACE từng bước (mô phỏng) để thấy bất biến

Bảng cho `n = 14`. Cột cuối là **bất biến** `teams + total`: luôn bằng `n`! Vì mỗi trận làm `total` tăng 1 và `teams` giảm 1.

| Vòng | teams trước | chẵn/lẻ | trận | teams sau | total | teams + total |
|---:|---:|---|---:|---:|---:|---:|
| 1 | 14 | chẵn | 7 | 7 | 7 | 14 |
| 2 | 7 | lẻ | 3 | 4 | 10 | 14 |
| 3 | 4 | chẵn | 2 | 2 | 12 | 14 |
| 4 | 2 | chẵn | 1 | 1 | 13 | 14 |

Khi `teams = 1`: `total = 14 - 1 = 13`.

In [13]:
def trace_tournament(n: int) -> int:
    teams, total, rnd = n, 0, 0
    print(f"Bắt đầu: teams = {teams}, total = {total}, teams + total = {teams + total}")
    while teams > 1:
        rnd += 1
        parity = "chẵn" if teams % 2 == 0 else "lẻ"
        matches = teams // 2
        before = teams
        teams = (teams + 1) // 2
        total += matches
        bye = " (1 đội đặc cách)" if parity == "lẻ" else ""
        print(f"Vòng {rnd}: {before:>3} đội ({parity}){bye} -> {matches} trận -> còn {teams}, "
              f"total = {total}, teams + total = {teams + total}")
    print("Kết quả:", total)
    return total

trace_tournament(14)
print()
trace_tournament(7)

Bắt đầu: teams = 14, total = 0, teams + total = 14
Vòng 1:  14 đội (chẵn) -> 7 trận -> còn 7, total = 7, teams + total = 14
Vòng 2:   7 đội (lẻ) (1 đội đặc cách) -> 3 trận -> còn 4, total = 10, teams + total = 14
Vòng 3:   4 đội (chẵn) -> 2 trận -> còn 2, total = 12, teams + total = 14
Vòng 4:   2 đội (chẵn) -> 1 trận -> còn 1, total = 13, teams + total = 14
Kết quả: 13

Bắt đầu: teams = 7, total = 0, teams + total = 7
Vòng 1:   7 đội (lẻ) (1 đội đặc cách) -> 3 trận -> còn 4, total = 3, teams + total = 7
Vòng 2:   4 đội (chẵn) -> 2 trận -> còn 2, total = 5, teams + total = 7
Vòng 3:   2 đội (chẵn) -> 1 trận -> còn 1, total = 6, teams + total = 7
Kết quả: 6


6

## 7. Lời giải khác — đệ quy

Định nghĩa `f(n)` = số trận cần cho `n` đội.
- Cơ sở: `f(1) = 0`.
- Đệ quy: vòng này có `n // 2` trận, còn `(n + 1) // 2` đội → `f(n) = n // 2 + f((n + 1) // 2)`.

In [14]:
def matches_recursive(n: int) -> int:
    if n == 1:
        return 0
    return n // 2 + matches_recursive((n + 1) // 2)

print([matches_recursive(n) for n in [1, 2, 7, 14, 200]])

[0, 1, 6, 13, 199]


Đệ quy sâu `O(log n)` tầng → Space `O(log n)` cho call stack, Time `O(log n)`.

### Bảng so sánh các cách

| Cách | Ý tưởng | Time | Space | Ghi chú |
|---|---|---|---|---|
| Mô phỏng chi tiết | đúng từng câu của đề | `O(log n)` | `O(1)` | dễ hiểu nhất |
| Mô phỏng gọn | `n//2` trận, `(n+1)//2` đội | `O(log n)` | `O(1)` | gọn hơn |
| Đệ quy | `f(n) = n//2 + f((n+1)//2)` | `O(log n)` | `O(log n)` | luyện tư duy đệ quy |
| Công thức | mỗi trận loại 1 đội | `O(1)` | `O(1)` | tối ưu, cần insight |

## 8. Code nộp LeetCode

Bản tối ưu:

In [15]:
class Solution:
    def numberOfMatches(self, n: int) -> int:
        return n - 1

print(Solution().numberOfMatches(7), Solution().numberOfMatches(14))   # 6 13

6 13


Bản mô phỏng (nếu phỏng vấn yêu cầu "làm theo luật"):

In [16]:
class SolutionSimulation:
    def numberOfMatches(self, n: int) -> int:
        total = 0
        while n > 1:
            total += n // 2
            n = (n + 1) // 2
        return total

print(SolutionSimulation().numberOfMatches(7), SolutionSimulation().numberOfMatches(14))

6 13


## 9. Vì sao đúng?

**Mô phỏng:** mỗi vòng ta cài đặt đúng nguyên văn luật của đề (đã kiểm chứng `n // 2 == (n-1)//2` khi lẻ, và `(n + 1)//2` là số đội đi tiếp). Vòng lặp dừng vì `teams ≥ 2 ⇒ (teams + 1)//2 < teams` — số đội giảm ngặt.

**Công thức — bất biến:** `teams + total = n` luôn đúng:
- Ban đầu: `teams = n, total = 0`.
- Một vòng có `m = teams // 2` trận ⇒ `total += m`, và có đúng `m` đội thua bị loại ⇒ `teams -= m`. Tổng không đổi.
- Kết thúc `teams = 1` ⇒ `total = n - 1`.

Lập luận này **không phụ thuộc** cách ghép cặp, nên đúng cho mọi giải đấu loại trực tiếp (knock-out) một nhà vô địch.

## 10. Độ phức tạp

Gọi `n` là số đội.
- **Công thức:** Time `O(1)`, Space `O(1)`.
- **Mô phỏng:** số vòng `⌈log₂ n⌉` ⇒ Time `O(log n)`, Space `O(1)`.
- **Đệ quy:** Time `O(log n)`, Space `O(log n)` (call stack).

In [17]:
import math
for n in [2, 7, 14, 200, 10**6]:
    rounds, t = 0, n
    while t > 1:
        t = (t + 1) // 2
        rounds += 1
    print(f"n = {n:>8}: số vòng = {rounds:>2}, ceil(log2 n) = {math.ceil(math.log2(n))}")

n =        2: số vòng =  1, ceil(log2 n) = 1
n =        7: số vòng =  3, ceil(log2 n) = 3
n =       14: số vòng =  4, ceil(log2 n) = 4
n =      200: số vòng =  8, ceil(log2 n) = 8
n =  1000000: số vòng = 20, ceil(log2 n) = 20


## 11. Bộ test

1. Ví dụ đề: `7 → 6`, `14 → 13`.
2. Biên: `n = 1` (0 trận), `n = 2` (1 trận), `n = 200` (giới hạn trên).
3. Trường hợp dễ sai: `n` lẻ (đặc cách), lũy thừa của 2 (không bao giờ đặc cách), `n = 3`.
4. Đối chiếu mọi cách với `n` từ 1 đến 2000.

In [18]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"PASS {name}: {actual}")

S = Solution()
check("đề 1 (n=7)", S.numberOfMatches(7), 6)
check("đề 2 (n=14)", S.numberOfMatches(14), 13)
check("n=1", S.numberOfMatches(1), 0)
check("n=2", S.numberOfMatches(2), 1)
check("n=3 lẻ", S.numberOfMatches(3), 2)
check("n=16 lũy thừa 2", S.numberOfMatches(16), 15)
check("n=200", S.numberOfMatches(200), 199)

PASS đề 1 (n=7): 6
PASS đề 2 (n=14): 13
PASS n=1: 0
PASS n=2: 1
PASS n=3 lẻ: 2
PASS n=16 lũy thừa 2: 15
PASS n=200: 199


In [19]:
impls = [matches_simulation, matches_simulation_compact, matches_recursive, matches_formula,
         eliminated_count, Solution().numberOfMatches, SolutionSimulation().numberOfMatches]
for n in range(1, 2001):
    vals = {f(n) for f in impls}
    assert len(vals) == 1, (n, vals)
print("7 cài đặt khớp nhau với n = 1..2000: PASS")

7 cài đặt khớp nhau với n = 1..2000: PASS


## 12. Lỗi người mới hay gặp

### Lỗi 1 — Quên đội đặc cách khi lẻ

Viết `teams = teams // 2` cho cả trường hợp lẻ → mất 1 đội mỗi lần lẻ.

In [20]:
def wrong_forget_bye(n):
    total = 0
    while n > 1:
        total += n // 2
        n = n // 2          # SAI khi n lẻ: phải là (n + 1) // 2
    return total

for n in [7, 14, 5]:
    print(n, "sai:", wrong_forget_bye(n), "| đúng:", matches_formula(n))

7 sai: 4 | đúng: 6
14 sai: 11 | đúng: 13
5 sai: 3 | đúng: 4


**Sửa:** số đội đi tiếp là `(n + 1) // 2` (hoặc `(n - 1)//2 + 1` khi lẻ).

### Lỗi 2 — Dùng `/` thay cho `//`

`/` sinh `float`: với `n` lẻ ra `3.5` trận — vô nghĩa. Tệ hơn, `n = (n + 1) / 2` tiến dần về `1.0` từ phía trên (`4.0 → 2.5 → 1.75 → 1.375 → ...`) nên vòng lặp chạy thêm rất nhiều vòng thừa và cộng dồn các 'nửa trận' cho tới khi số thực làm tròn đúng bằng `1.0`. Kết quả là một số thực lộn xộn.

In [21]:
def wrong_float(n):
    total = 0
    while n > 1:
        total += n / 2        # SAI: float, và 7/2 = 3.5
        n = (n + 1) / 2
    return total

print("sai:", wrong_float(7), "| đúng:", matches_formula(7))

sai: 34.0 | đúng: 6


**Sửa:** luôn dùng `//` khi đếm số lượng nguyên.

### Lỗi 3 — Điều kiện dừng `while n > 0`

Khi `n = 1`, `(1 + 1)//2 = 1` → vòng lặp không bao giờ kết thúc! (Ta không chạy code này, chỉ minh hoạ bằng cách giới hạn số vòng.)

In [22]:
def wrong_stop(n, max_rounds=10):
    total, rounds = 0, 0
    while n > 0 and rounds < max_rounds:     # n > 0 là SAI; max_rounds chỉ để demo không treo
        total += n // 2
        n = (n + 1) // 2
        rounds += 1
    return n, rounds

print("(n cuối, số vòng đã chạy):", wrong_stop(7), "-> bị kẹt ở n = 1 mãi mãi")

(n cuối, số vòng đã chạy): (1, 10) -> bị kẹt ở n = 1 mãi mãi


**Sửa:** dừng khi chỉ còn **một** đội: `while n > 1`.

### Lỗi 4 — Trả `n // 2` (chỉ đếm vòng đầu)

Một số bạn đọc lướt và chỉ tính trận của vòng đầu tiên: `7 // 2 = 3` ≠ 6. Phải cộng **tất cả** các vòng.

## 13. Mẹo nhận dạng pattern

- Đề "đếm tổng qua nhiều vòng mô phỏng" với số liệu nhỏ → mô phỏng trước, **in bảng kết quả theo `n`** để tìm quy luật.
- Tìm **đại lượng bất biến** (ở đây `teams + total`) hoặc **đếm theo đối tượng khác** (đếm đội bị loại thay vì đếm trận) — kỹ thuật "double counting".
- Giải loại trực tiếp: `n` đội, 1 vô địch ⇒ `n - 1` trận. Nếu cần `k` người sống sót ⇒ `n - k` trận.
- Bài liên quan: 292 Nim Game, 319 Bulb Switcher (quan sát quy luật rồi chứng minh), 1025 Divisor Game.

## 14. Biến thể

### Biến thể 1 — Mỗi trận có 3 đội, chỉ 1 đội thắng

Mỗi trận loại 2 đội. Nếu luôn kết thúc được (cho phép đặc cách 1–2 đội), số trận = `(n - 1) / 2`. Mô phỏng để kiểm chứng với `n` lẻ:

In [23]:
def matches_triple(n):
    total = 0
    while n > 1:
        if n == 2:              # chỉ còn 2 đội: đá 1 trận đôi (giả định)
            total += 1
            n = 1
            continue
        m = n // 3
        total += m
        n = n - 2 * m            # mỗi trận loại 2 đội
    return total

for n in [3, 5, 7, 9, 27]:
    print(n, "->", matches_triple(n), "| (n - 1) // 2 =", (n - 1) // 2)

3 -> 1 | (n - 1) // 2 = 1
5 -> 2 | (n - 1) // 2 = 2
7 -> 3 | (n - 1) // 2 = 3
9 -> 4 | (n - 1) // 2 = 4
27 -> 13 | (n - 1) // 2 = 13


### Biến thể 2 — In lịch thi đấu từng vòng

Thay vì chỉ đếm, trả về danh sách `(số đội, số trận, có đặc cách?)` mỗi vòng.

In [24]:
def schedule(n):
    rounds = []
    while n > 1:
        rounds.append((n, n // 2, n % 2 == 1))
        n = (n + 1) // 2
    return rounds

for r in schedule(13):
    print(f"{r[0]:>3} đội -> {r[1]} trận, đặc cách: {r[2]}")

 13 đội -> 6 trận, đặc cách: True
  7 đội -> 3 trận, đặc cách: True
  4 đội -> 2 trận, đặc cách: False
  2 đội -> 1 trận, đặc cách: False


### Biến thể 3 — Đếm số lần có đội đặc cách

Đặc cách xảy ra khi `teams` lẻ (và > 1).

In [25]:
def count_byes(n):
    return sum(1 for teams, _, bye in schedule(n) if bye)

print([count_byes(n) for n in [1, 7, 8, 13, 200]])

[0, 1, 0, 2, 3]


## 15. Bài tập (tăng dần)

> Các hàm bài tập chỉ có `pass`, **không được gọi** khi chạy notebook.

### Bài 1 — Tính tay
Với `n = 11`, lập bảng từng vòng (số đội, trận, đặc cách) và tổng số trận.

In [26]:
def exercise_1_rounds():
    # TODO: trả về list các số trận của từng vòng với n = 11, ví dụ [.., .., ..]
    pass

# assert sum(exercise_1_rounds()) == 10

### Bài 2 — Điền chỗ trống mô phỏng

In [27]:
def simulate_fill(n: int) -> int:
    total = 0
    # TODO 1: lặp khi còn hơn 1 đội
    # TODO 2: cộng số trận vòng này vào total
    # TODO 3: cập nhật số đội đi tiếp
    pass

# assert simulate_fill(7) == 6 and simulate_fill(1) == 0

### Bài 3 — Đếm số vòng đấu
Viết `count_rounds(n)` trả về số vòng (với `n = 1` là 0).

In [28]:
def count_rounds(n: int) -> int:
    # TODO
    pass

# assert count_rounds(7) == 3 and count_rounds(14) == 4 and count_rounds(1) == 0

### Bài 4 — Biến thể: còn `k` đội sống sót
Giải dừng khi còn **`k` đội** (`1 <= k <= n`). Trả về số trận (mỗi trận loại 1 đội).

In [29]:
def matches_until_k(n: int, k: int) -> int:
    # TODO: dùng lập luận đếm đội bị loại
    pass

# assert matches_until_k(14, 1) == 13 and matches_until_k(14, 4) == 10 and matches_until_k(5, 5) == 0

### Bài 5 — Đệ quy đếm đặc cách
Viết `byes_recursive(n)` bằng đệ quy (không dùng vòng lặp).

In [30]:
def byes_recursive(n: int) -> int:
    # TODO: cơ sở n == 1; bước: (1 nếu n lẻ) + byes_recursive((n + 1) // 2)
    pass

# assert byes_recursive(13) == count_byes(13)

### Bài 6 — Mini challenge: tìm `n` nhỏ nhất có nhiều đặc cách nhất
Trong `2 <= n <= 200`, tìm `n` nhỏ nhất có số lần đặc cách lớn nhất.

In [31]:
def most_byes(limit: int = 200) -> int:
    # TODO
    pass

# assert most_byes(200) == ...  # tự kiểm chứng bằng count_byes

## 16. Gợi ý 3 mức

**Bài 1:** (1) 11 lẻ → đặc cách. (2) Mỗi vòng: trận = `n // 2`, đội = `(n + 1)//2`. (3) 11→6→3→2→1, trận `5, 3, 1, 1`.

**Bài 2:** (1) Giống `matches_simulation_compact`. (2) `while n > 1`. (3) `total += n // 2; n = (n + 1) // 2`, cuối `return total`.

**Bài 3:** (1) Thêm biến đếm vòng. (2) Tăng 1 mỗi lần lặp. (3) Kết quả xấp xỉ `ceil(log2 n)`.

**Bài 4:** (1) Mỗi trận loại đúng 1 đội. (2) Cần loại bao nhiêu đội để còn `k`? (3) `n - k`.

**Bài 5:** (1) Cơ sở `n == 1 → 0`. (2) `n % 2` cho biết có đặc cách. (3) `n % 2 + byes_recursive((n + 1)//2)`.

**Bài 6:** (1) Duyệt mọi `n`. (2) Giữ `best_n`, `best_cnt`. (3) Chỉ cập nhật khi `count_byes(n) > best_cnt` (dấu `>` để lấy `n` nhỏ nhất).

## 17. Đáp án tham khảo

In [32]:
def ans_exercise_1_rounds():
    n, res = 11, []
    while n > 1:
        res.append(n // 2)
        n = (n + 1) // 2
    return res

def ans_simulate_fill(n):
    total = 0
    while n > 1:
        total += n // 2
        n = (n + 1) // 2
    return total

def ans_count_rounds(n):
    r = 0
    while n > 1:
        n = (n + 1) // 2
        r += 1
    return r

def ans_matches_until_k(n, k):
    return n - k

def ans_byes_recursive(n):
    if n == 1:
        return 0
    return n % 2 + ans_byes_recursive((n + 1) // 2)

def ans_most_byes(limit=200):
    best_n, best_cnt = 2, -1
    for n in range(2, limit + 1):
        c = count_byes(n)
        if c > best_cnt:
            best_n, best_cnt = n, c
    return best_n

print("Bài 1:", ans_exercise_1_rounds())
assert ans_exercise_1_rounds() == [5, 3, 1, 1] and sum(ans_exercise_1_rounds()) == 10
assert ans_simulate_fill(7) == 6 and ans_simulate_fill(1) == 0
assert ans_count_rounds(7) == 3 and ans_count_rounds(14) == 4 and ans_count_rounds(1) == 0
assert ans_matches_until_k(14, 1) == 13 and ans_matches_until_k(14, 4) == 10 and ans_matches_until_k(5, 5) == 0
assert all(ans_byes_recursive(n) == count_byes(n) for n in range(1, 300))
print("Bài 6: n =", ans_most_byes(200), "với", count_byes(ans_most_byes(200)), "lần đặc cách")
print("Tất cả đáp án PASS!")

Bài 1: [5, 3, 1, 1]
Bài 6: n = 129 với 7 lần đặc cách
Tất cả đáp án PASS!


## 18. Template nhớ nhanh + checklist submit

```python
class Solution:
    def numberOfMatches(self, n: int) -> int:
        return n - 1            # mỗi trận loại đúng 1 đội, cần loại n - 1 đội
```

Mô phỏng dự phòng:

```python
total = 0
while n > 1:
    total += n // 2
    n = (n + 1) // 2
```

**Khung tư duy:** mô phỏng đúng luật → in bảng theo `n` → đoán quy luật → **chứng minh bằng bất biến / đếm theo đối tượng khác**.

**Checklist trước khi submit**
- [ ] `n = 1` trả về `0`?
- [ ] Trường hợp lẻ có cộng đội đặc cách (`(n + 1) // 2`)?
- [ ] Dùng `//`, không dùng `/`?
- [ ] Điều kiện dừng `n > 1` (không phải `n > 0`)?
- [ ] Tên method `numberOfMatches`, trả về `int`?